# Colab Run – TS2IMG LightCNN Experiments with Resume

Notebook này hỗ trợ chạy tiếp khi Colab bị dừng đột ngột.

Nguyên tắc:
- Muốn làm lại từ đầu: bật `CONFIRM_FRESH_START = True` ở cell xóa kết quả.
- Muốn chạy tiếp sau khi Colab bị dừng: không chạy cell xóa kết quả.
- Lệnh chính thức dùng `--resume` để bỏ qua các thí nghiệm đã hoàn thành trong `results/summary_results.csv`.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone hoặc cập nhật repo từ GitHub

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/hoangtnedu/ts2img-lightcnn.git"
REPO_DIR = Path("/content/drive/MyDrive/ts2img-lightcnn")
BRANCH = "main"

if (REPO_DIR / ".git").exists():
    print(f"Repo already exists: {REPO_DIR}")
    %cd {REPO_DIR}
    !git fetch origin
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
else:
    print(f"Cloning repo to: {REPO_DIR}")
    %cd /content/drive/MyDrive
    !git clone {REPO_URL}
    %cd {REPO_DIR}

!pwd
!git log --oneline -1

## 3. Cài đặt thư viện

In [ ]:
%cd /content/drive/MyDrive/ts2img-lightcnn
!python -m pip install --upgrade pip
!pip install -r requirements.txt

## 4. Kiểm tra GPU và code resume

In [ ]:
%cd /content/drive/MyDrive/ts2img-lightcnn

import tensorflow as tf
import numpy as np
import pandas as pd
import aeon

print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

print("\nCheck aeon:")
!grep -n "aeon" requirements.txt || true
!grep -n "load_classification" src/data_ucr.py || true

print("\nCheck resume:")
!grep -n "resume" src/run_experiments.py || true
!grep -n "SKIP completed" src/run_experiments.py || true

## 5. Test loader 5 bộ dữ liệu

In [ ]:
from src.data_ucr import load_dataset

datasets = ["GunPoint", "ECG200", "Coffee", "FordA", "Wafer"]

for name in datasets:
    X_train, X_test, y_train, y_test, num_classes, encoder = load_dataset(name)
    print(f"{name:10s} | X_train={X_train.shape} | X_test={X_test.shape} | classes={num_classes}")

## 6. Kiểm tra kết quả hiện có

In [ ]:
from pathlib import Path
import pandas as pd

result_path = Path("results/summary_results.csv")

if result_path.exists():
    df = pd.read_csv(result_path)
    print("Existing result file:", result_path)
    print("Shape:", df.shape)
    print("\nDataset counts:")
    print(df["dataset"].value_counts())
    print("\nRepresentation counts:")
    print(df["representation"].value_counts())
    print("\nSeed counts:")
    print(df["seed"].value_counts())
    display(df.tail(10))
else:
    print("No existing results/summary_results.csv found.")

## 7. Tùy chọn: Xóa kết quả để chạy lại từ đầu

In [ ]:
%cd /content/drive/MyDrive/ts2img-lightcnn

CONFIRM_FRESH_START = False  # Chỉ đổi thành True khi muốn xóa sạch và chạy lại từ đầu

if CONFIRM_FRESH_START:
    !rm -rf runs/*
    !rm -f results/summary_results.csv
    print("Old runs and summary_results.csv were removed.")
else:
    print("Fresh start is disabled. Nothing was deleted.")

## 8. Test nhanh pipeline có resume

In [ ]:
%cd /content/drive/MyDrive/ts2img-lightcnn

!python -m src.run_experiments \
  --datasets GunPoint,ECG200,Coffee,FordA,Wafer \
  --representations gaf,rp \
  --model_type light2dcnn \
  --seeds 42 \
  --epochs 2 \
  --batch_size 16 \
  --image_size 64 \
  --resume

## 9. Chạy chính thức có resume

In [ ]:
%cd /content/drive/MyDrive/ts2img-lightcnn

!python -m src.run_experiments \
  --datasets GunPoint,ECG200,Coffee,FordA,Wafer \
  --representations gaf,mtf,rp,stft \
  --model_type light2dcnn \
  --seeds 42,2024,2026 \
  --epochs 50 \
  --batch_size 32 \
  --image_size 64 \
  --resume

## 10. Kiểm tra kết quả chính thức

In [ ]:
import pandas as pd

df = pd.read_csv("results/summary_results.csv")

official = df[
    (df["epochs_requested"] == 50) &
    (df["batch_size"] == 32) &
    (df["image_size"] == 64)
].copy()

print("All rows:", df.shape)
print("Official rows:", official.shape)

print("\nDataset counts:")
print(official["dataset"].value_counts())

print("\nRepresentation counts:")
print(official["representation"].value_counts())

print("\nSeed counts:")
print(official["seed"].value_counts())

print("\nExpected official result:")
print("Total rows: 75; each dataset: 15; each representation: 15; each seed: 25")

display(official.tail(20))

## 11. Tạo bảng tổng hợp cho bài báo

In [ ]:
import pandas as pd

df = pd.read_csv("results/summary_results.csv")

official = df[
    (df["epochs_requested"] == 50) &
    (df["batch_size"] == 32) &
    (df["image_size"] == 64)
].copy()

official["method"] = official.apply(
    lambda r: "1D-CNN" if r["representation"] == "none"
    else r["representation"].upper() + " + Light 2D-CNN",
    axis=1
)

summary = official.groupby(["dataset", "method"]).agg(
    accuracy_mean=("accuracy", "mean"),
    accuracy_std=("accuracy", "std"),
    macro_f1_mean=("macro_f1", "mean"),
    macro_f1_std=("macro_f1", "std"),
    params_mean=("params", "mean"),
    train_time_mean=("training_time_sec", "mean"),
    infer_time_mean=("inference_time_per_sample_sec", "mean")
).reset_index()

summary.to_csv("results/summary_by_dataset_method.csv", index=False)
display(summary)

## 12. Tạo bảng Accuracy / Macro F1-score

In [ ]:
paper_table_source = summary.copy()
paper_table_source["result"] = paper_table_source.apply(
    lambda r: f"{r['accuracy_mean']:.4f} / {r['macro_f1_mean']:.4f}",
    axis=1
)

paper_table = paper_table_source.pivot(
    index="dataset",
    columns="method",
    values="result"
).reset_index()

preferred_columns = [
    "dataset",
    "1D-CNN",
    "GAF + Light 2D-CNN",
    "MTF + Light 2D-CNN",
    "RP + Light 2D-CNN",
    "STFT + Light 2D-CNN",
]

paper_table = paper_table[[c for c in preferred_columns if c in paper_table.columns]]
paper_table.to_csv("results/table_accuracy_f1_for_paper.csv", index=False)
display(paper_table)

## 13. Tạo bảng phương pháp tốt nhất và file ZIP kết quả

In [ ]:
best = summary.sort_values(
    ["dataset", "macro_f1_mean", "accuracy_mean"],
    ascending=[True, False, False]
).groupby("dataset").head(1)

best.to_csv("results/best_method_by_dataset.csv", index=False)
display(best)

!zip -r results_colab.zip results runs -x "runs/**/*.keras"
print("Saved: results_colab.zip")